# LangChain 기초 전체 학습 노트북

이 노트북은 `week02/practice`에서 LangChain을 처음 배우는 사람이 혼자 따라가며 공부할 수 있도록 만든 입문 자료입니다.

LangChain은 LLM 앱을 만들 때 자주 반복되는 작업을 표준화합니다. 예를 들어 프롬프트 만들기, 모델 호출하기, 출력 파싱하기, 문서 자르기, 벡터 검색하기, 도구를 붙여 agent 만들기 같은 흐름을 하나의 방식으로 연결할 수 있게 도와줍니다.

## 이 노트북에서 배우는 것

- LangChain 1.x의 큰 구조와 패키지 역할
- 메시지, 프롬프트, 모델, 출력 파서의 기본 사용법
- LCEL과 Runnable로 체인을 연결하는 방법
- `invoke`, `batch`, `stream` 실행 방식
- `Document`, loader, splitter, embedding, vector store, retriever의 역할
- 아주 작은 RAG 파이프라인을 만드는 방법
- tool과 agent의 기본 개념
- 처음 배울 때 자주 헷갈리는 부분과 점검 질문

> 대부분의 실습은 API 키 없이 실행되도록 fake model과 작은 예제 데이터를 사용합니다. 실제 LLM을 호출하는 셀은 `OPENAI_API_KEY`가 있을 때만 실행되도록 작성했습니다.


## 0. 추천 학습 순서

처음 공부한다면 아래 순서로 진행하세요.

1. 먼저 `LangChain이 해결하는 문제`를 말로 이해합니다.
2. `prompt -> model -> parser`의 가장 작은 체인을 실행해 봅니다.
3. LCEL의 `|` 연산자가 앞 단계의 출력을 다음 단계로 넘긴다는 점을 확인합니다.
4. 문서를 `Document -> split -> embedding -> vector store -> retriever` 흐름으로 검색해 봅니다.
5. 마지막에 RAG와 agent를 비교하면서 어디에 LangChain을 쓰는지 정리합니다.

중요한 목표는 모든 API를 외우는 것이 아니라, **LLM 앱의 부품들이 어떤 순서로 연결되는지**를 이해하는 것입니다.


## 1. 환경 확인

현재 프로젝트의 `requirements.txt`는 LangChain 1.x 계열을 사용합니다. LangChain은 버전 변화가 빠르므로, 코드가 잘 안 될 때는 먼저 버전을 확인하는 습관이 중요합니다.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version

packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-openai",
    "langchain-text-splitters",
    "langchain-chroma",
]

print(sys.version)
for package in packages:
    try:
        print(f"{package:24s}", version(package))
    except PackageNotFoundError:
        print(f"{package:24s}", "not installed")


## 2. LangChain의 큰 그림

LangChain을 처음 보면 import 경로가 많아서 어렵게 느껴집니다. 하지만 역할별로 보면 단순합니다.

| 역할 | 주로 쓰는 패키지 | 예시 |
|---|---|---|
| 핵심 추상화 | `langchain_core` | `ChatPromptTemplate`, `Document`, `Runnable`, `StrOutputParser` |
| 모델 provider 연동 | `langchain_openai`, 기타 provider 패키지 | `ChatOpenAI`, `OpenAIEmbeddings` |
| 문서 로더 등 커뮤니티 통합 | `langchain_community` | `PyPDFLoader`, `WebBaseLoader` |
| 텍스트 분할 | `langchain_text_splitters` | `RecursiveCharacterTextSplitter` |
| 벡터 저장소 연동 | `langchain_chroma`, provider별 패키지 | `Chroma` |
| agent 생성 | `langchain` | `create_agent` |

초보자는 먼저 아래 흐름만 기억하면 됩니다.

```text
입력 질문
  -> Prompt
  -> Model
  -> Output Parser
  -> 최종 답변
```

RAG에서는 이 앞에 문서 검색 단계가 붙습니다.

```text
문서
  -> Split
  -> Embedding
  -> Vector Store
  -> Retriever
  -> Prompt
  -> Model
  -> Answer
```


## 3. Message: LLM 대화의 기본 단위

Chat model은 보통 문자열 하나가 아니라 메시지 목록을 입력으로 받습니다.

- `SystemMessage`: 모델의 역할, 규칙, 말투를 지정
- `HumanMessage`: 사용자의 입력
- `AIMessage`: 모델의 이전 답변

실제 대화형 앱에서는 이 메시지 목록이 대화 기록이 됩니다.


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 초보자에게 친절하게 설명하는 LangChain 튜터야."),
    HumanMessage(content="LangChain이 왜 필요한지 한 문장으로 설명해줘."),
]

for message in messages:
    print(type(message).__name__, "->", message.content)


## 4. PromptTemplate: 입력값을 프롬프트로 바꾸기

프롬프트는 매번 손으로 문자열을 붙이는 대신 template으로 관리하는 것이 좋습니다.

`PromptTemplate`은 일반 텍스트 프롬프트에 쓰고, `ChatPromptTemplate`은 chat model에 넣을 메시지 목록을 만들 때 씁니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

string_prompt = PromptTemplate.from_template(
    "{topic}을 처음 배우는 사람에게 비유를 들어 설명해줘."
)

prompt_value = string_prompt.invoke({"topic": "벡터 데이터베이스"})
print(prompt_value.text)


In [ ]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 {audience}에게 쉽게 설명하는 선생님이야."),
        ("human", "{question}"),
    ]
)

chat_prompt_value = chat_prompt.invoke(
    {
        "audience": "LangChain 입문자",
        "question": "프롬프트 템플릿이 왜 필요해?",
    }
)

for message in chat_prompt_value.messages:
    print(type(message).__name__, "->", message.content)


## 5. Model: 실제 LLM 또는 Fake Model 호출하기

LangChain에서 model은 `invoke()`로 호출할 수 있는 Runnable입니다. 이 노트북에서는 API 키 없이도 학습할 수 있도록 `FakeListChatModel`을 먼저 사용합니다.

Fake model은 미리 넣어 둔 답변을 그대로 반환합니다. 지능이 있는 모델은 아니지만, LangChain 체인의 모양을 이해하기에 좋습니다.


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake_model = FakeListChatModel(
    responses=["LangChain은 프롬프트, 모델, 검색, 도구를 연결해 LLM 앱을 쉽게 만드는 프레임워크입니다."]
)

response = fake_model.invoke(messages)
print(type(response).__name__)
print(response.content)


### 실제 모델 호출은 선택 사항

아래 셀은 `OPENAI_API_KEY`가 있을 때만 실행됩니다. 모델 이름은 환경 변수 `LANGCHAIN_MODEL`로 바꿀 수 있고, 기본값은 공식 문서 예시와 같은 provider 문자열 형식입니다.

처음 공부할 때는 이 셀을 건너뛰어도 됩니다. 핵심은 LangChain의 연결 구조입니다.


In [ ]:
import os

if os.environ.get("OPENAI_API_KEY"):
    from langchain.chat_models import init_chat_model

    model_name = os.environ.get("LANGCHAIN_MODEL", "openai:gpt-5.4")
    real_model = init_chat_model(model_name, temperature=0)
    real_response = real_model.invoke("LangChain을 한 문장으로 설명해줘.")
    print(real_response.content)
else:
    print("OPENAI_API_KEY가 없어서 실제 모델 호출은 건너뜁니다.")


## 6. Output Parser: 모델 출력을 원하는 형태로 바꾸기

모델 응답은 보통 `AIMessage`입니다. 앱에서는 문자열, JSON, Pydantic 객체처럼 더 다루기 쉬운 형태가 필요합니다.

가장 기본은 `StrOutputParser`입니다. `AIMessage.content`만 꺼내 문자열로 바꿔 줍니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parsed = parser.invoke(response)
print(type(parsed).__name__)
print(parsed)


JSON처럼 구조화된 출력이 필요할 때는 `JsonOutputParser`를 사용할 수 있습니다. 실제 모델을 쓸 때는 parser가 제공하는 format instruction을 프롬프트에 넣어 주는 것이 중요합니다.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class StudyPlan(BaseModel):
    topic: str = Field(description="학습 주제")
    steps: list[str] = Field(description="공부 순서")
    difficulty: int = Field(description="1부터 5까지의 난이도")

json_parser = JsonOutputParser(pydantic_object=StudyPlan)
print(json_parser.get_format_instructions()[:500])


In [ ]:
json_fake_model = FakeListChatModel(
    responses=[
        '{"topic": "LCEL", "steps": ["Prompt", "Model", "Parser"], "difficulty": 2}'
    ]
)

json_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 학습 계획을 JSON으로만 답하는 튜터야."),
        ("human", "{topic} 공부 계획을 만들어줘.\n{format_instructions}"),
    ]
)

json_chain = json_prompt.partial(
    format_instructions=json_parser.get_format_instructions()
) | json_fake_model | json_parser

json_chain.invoke({"topic": "LangChain Expression Language"})


## 7. LCEL: `|`로 체인 연결하기

LCEL은 LangChain Expression Language의 약자입니다. 가장 많이 보는 형태는 아래처럼 `|`로 Runnable들을 연결하는 방식입니다.

```python
chain = prompt | model | parser
```

의미는 단순합니다.

```text
prompt의 출력 -> model의 입력 -> parser의 입력 -> 최종 출력
```


In [ ]:
explain_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 어려운 개념을 쉬운 한국어로 설명하는 선생님이야."),
        ("human", "{concept}을 2문장으로 설명해줘."),
    ]
)

explain_model = FakeListChatModel(
    responses=["LCEL은 여러 단계를 파이프처럼 이어 주는 LangChain의 표현 방식입니다. 앞 단계의 결과가 다음 단계의 입력으로 넘어갑니다."]
)

explain_chain = explain_prompt | explain_model | StrOutputParser()
print(explain_chain.invoke({"concept": "LCEL"}))


## 8. Runnable의 공통 실행 방식

LangChain의 많은 객체는 Runnable입니다. Runnable은 공통적으로 다음 메서드를 제공합니다.

- `invoke(input)`: 입력 하나를 처리
- `batch([input1, input2])`: 여러 입력을 한 번에 처리
- `stream(input)`: 가능한 경우 토큰이나 조각 단위로 출력
- `with_config(...)`: tags, metadata 등 실행 설정 추가

이 공통 인터페이스 덕분에 prompt, model, parser, retriever를 비슷한 방식으로 다룰 수 있습니다.


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

normalize = RunnableLambda(lambda text: text.strip().lower())
count_chars = RunnableLambda(len)

text_pipeline = normalize | RunnableParallel(
    normalized=RunnablePassthrough(),
    length=count_chars,
)

text_pipeline.invoke("  LangChain Basics  ")


In [ ]:
text_pipeline.batch(
    [
        "  Prompt  ",
        "  Model  ",
        "  Retriever  ",
    ]
)


In [ ]:
stream_model = FakeListChatModel(responses=["stream은 답변이 만들어지는 조각을 순서대로 받을 때 사용합니다."])

for chunk in stream_model.stream("stream 예시를 보여줘"):
    print(chunk.content, end="")


## 9. Chat History: 이전 대화 넣기

LLM은 자동으로 대화를 기억하지 않습니다. 이전 대화를 기억하게 만들려면 메시지 목록을 다시 넣어야 합니다.

`MessagesPlaceholder`는 프롬프트 중간에 이전 대화 목록을 끼워 넣을 때 사용합니다.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder

history_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절한 학습 도우미야."),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

history_prompt_value = history_prompt.invoke(
    {
        "history": [
            HumanMessage(content="LangChain에서 prompt를 배웠어."),
            AIMessage(content="좋아요. 다음에는 model과 parser를 연결해 보면 됩니다."),
        ],
        "question": "그 다음에는 무엇을 공부하면 돼?",
    }
)

for message in history_prompt_value.messages:
    print(type(message).__name__, "->", message.content)


## 10. Document: 검색 대상 문서의 기본 단위

RAG에서 검색할 문서는 보통 `Document` 객체로 다룹니다.

`Document`는 크게 두 부분을 가집니다.

- `page_content`: 실제 텍스트
- `metadata`: 출처, 페이지, 카테고리, 날짜 같은 부가 정보

메타데이터는 나중에 필터링, 출처 표시, 권한 관리에 중요합니다.


In [ ]:
from langchain_core.documents import Document

sample_docs = [
    Document(
        page_content="LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크입니다.",
        metadata={"source": "study-note", "page": 1, "topic": "overview"},
    ),
    Document(
        page_content="Retriever는 질문과 관련 있는 문서를 찾아 prompt에 넣을 context를 준비합니다.",
        metadata={"source": "study-note", "page": 2, "topic": "retrieval"},
    ),
]

for doc in sample_docs:
    print(doc.page_content)
    print(doc.metadata)
    print("---")


## 11. Loader: 파일을 Document로 읽기

LangChain loader는 PDF, 웹페이지, 텍스트 파일 같은 원본 데이터를 `Document` 목록으로 바꿉니다.

아래 셀은 `weeks/week02/data/MIT.pdf`가 있으면 앞부분만 읽어 봅니다. 실행 위치가 repository root이든 노트북 폴더이든 찾을 수 있도록 후보 경로를 여러 개 둡니다.


In [ ]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader

candidate_paths = [
    Path("weeks/week02/data/MIT.pdf"),
    Path("../data/MIT.pdf"),
]

pdf_path = next((path for path in candidate_paths if path.exists()), None)

if pdf_path is None:
    print("MIT.pdf를 찾지 못했습니다. sample_docs로 다음 실습을 계속 진행합니다.")
    pdf_docs = []
else:
    loader = PyPDFLoader(str(pdf_path))
    pdf_docs = loader.load()
    print(f"loaded pages: {len(pdf_docs)}")
    print(pdf_docs[0].metadata)
    print(pdf_docs[0].page_content[:500])


## 12. Text Splitter: 긴 문서를 작은 chunk로 나누기

LLM과 retriever는 긴 문서 전체보다 적절히 나뉜 chunk를 다루는 경우가 많습니다.

`RecursiveCharacterTextSplitter`는 기본 splitter로 많이 사용됩니다. 문단, 문장, 단어 같은 자연스러운 경계를 최대한 유지하면서 chunk size에 맞춰 자릅니다.

중요 파라미터는 두 가지입니다.

- `chunk_size`: chunk 하나의 최대 길이
- `chunk_overlap`: 앞뒤 chunk가 겹치는 길이


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20,
)

chunks = splitter.split_documents(sample_docs)

for i, chunk in enumerate(chunks, start=1):
    print(f"chunk {i}")
    print(chunk.page_content)
    print(chunk.metadata)
    print("---")


## 13. Embedding: 텍스트를 숫자 벡터로 바꾸기

Embedding은 텍스트의 의미나 특징을 숫자 벡터로 바꾸는 단계입니다. 실제 서비스에서는 OpenAI, Hugging Face, Cohere 같은 embedding model을 사용합니다.

여기서는 API 키 없이 원리를 보기 위해 아주 단순한 학습용 embedding을 만듭니다. 정해 둔 단어들이 텍스트에 몇 번 등장하는지 세어 벡터로 바꾸는 방식입니다.

실제 semantic embedding만큼 똑똑하지는 않지만, `embedding -> vector store -> similarity search` 흐름을 이해하기에는 충분합니다.


In [ ]:
import math

from langchain_core.embeddings import Embeddings

class KeywordCountEmbeddings(Embeddings):
    def __init__(self, vocabulary: list[str]):
        self.vocabulary = [word.lower() for word in vocabulary]

    def _embed(self, text: str) -> list[float]:
        text_lower = text.lower()
        # 조사나 문장부호가 붙어도 `retriever는`, `RAG에서` 같은 표현을 잡기 위해 부분 문자열로 셉니다.
        vector = [float(text_lower.count(word)) for word in self.vocabulary]
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

vocabulary = [
    "langchain",
    "prompt",
    "model",
    "parser",
    "runnable",
    "document",
    "chunk",
    "embedding",
    "vector",
    "retriever",
    "rag",
    "agent",
    "tool",
    "metadata",
]

embedding_model = KeywordCountEmbeddings(vocabulary)
embedding_model.embed_query("prompt model parser")


## 14. Vector Store와 Similarity Search

Vector store는 문서 embedding을 저장하고, 질문과 비슷한 문서를 찾아 줍니다.

여기서는 로컬 메모리에서만 동작하는 `InMemoryVectorStore`를 사용합니다. 학습용으로는 충분하고, 실제 서비스에서는 Chroma, Pinecone, Qdrant, Milvus 같은 저장소를 사용할 수 있습니다.


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

rag_docs = [
    Document(
        page_content="LangChain connects prompt, model, parser, and runnable components.",
        metadata={"topic": "chain", "source": "mini-docs"},
    ),
    Document(
        page_content="RAG retrieves relevant document chunks before asking the model to answer.",
        metadata={"topic": "rag", "source": "mini-docs"},
    ),
    Document(
        page_content="A vector store saves embeddings and supports similarity search.",
        metadata={"topic": "vector-store", "source": "mini-docs"},
    ),
    Document(
        page_content="Agents can call tools when a model needs to take an action.",
        metadata={"topic": "agent", "source": "mini-docs"},
    ),
]

vector_store = InMemoryVectorStore(embedding=embedding_model)
vector_store.add_documents(rag_docs)

results = vector_store.similarity_search("vector embedding retriever", k=2)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")


## 15. Retriever: 검색 기능을 체인에 연결하기

Retriever는 질문을 받아 관련 `Document` 목록을 반환하는 객체입니다.

Vector store는 보통 `as_retriever()`로 retriever처럼 사용할 수 있습니다. 이렇게 만들면 나중에 RAG 체인에 자연스럽게 붙일 수 있습니다.


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

retrieved_docs = retriever.invoke("RAG document chunk")
for doc in retrieved_docs:
    print(doc.page_content)


## 16. Mini RAG Chain 만들기

이제 검색 결과를 prompt의 context로 넣어 보겠습니다.

진짜 RAG의 핵심은 아래 한 줄입니다.

```text
질문 -> 관련 문서 검색 -> 검색된 문서를 context로 prompt에 넣기 -> model 답변
```

여기서는 fake model을 사용하므로 답변은 미리 정해져 있습니다. 대신 체인의 구조와 데이터 흐름에 집중하세요.


In [ ]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[source={doc.metadata.get('source')}, topic={doc.metadata.get('topic')}]\n{doc.page_content}"
        for doc in docs
    )

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 주어진 context만 근거로 답하는 RAG assistant야."),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)

rag_model = FakeListChatModel(
    responses=["RAG는 질문과 관련 있는 문서 조각을 먼저 검색한 뒤, 그 내용을 근거로 모델이 답하게 만드는 방식입니다."]
)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | rag_model
    | StrOutputParser()
)

print(rag_chain.invoke("RAG에서 retriever는 무엇을 하나요?"))


In [ ]:
# RAG 체인에 들어가는 context가 실제로 어떻게 생겼는지 확인합니다.
context_only_chain = retriever | RunnableLambda(format_docs)
print(context_only_chain.invoke("RAG에서 retriever는 무엇을 하나요?"))


## 17. Tool: 모델이 호출할 수 있는 함수

Tool은 agent가 사용할 수 있는 함수입니다. 예를 들어 계산기, 검색 API, 데이터베이스 조회 함수가 tool이 될 수 있습니다.

중요한 점은 함수의 이름, 설명, 입력 타입이 모델에게 힌트가 된다는 것입니다. 그래서 docstring을 대충 쓰면 agent 품질도 떨어질 수 있습니다.


In [ ]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

print("name:", multiply.name)
print("description:", multiply.description)
print("result:", multiply.invoke({"a": 7, "b": 6}))


## 18. Agent: 모델이 tool 사용 여부를 결정하는 구조

Chain은 보통 정해진 순서대로 실행됩니다.

```text
prompt -> model -> parser
```

Agent는 모델이 상황을 보고 tool을 호출할지, 바로 답할지 결정합니다.

```text
user question -> model decides -> tool call maybe -> model final answer
```

아래 셀은 실제 모델 API 키가 있을 때만 실행됩니다. API 키가 없다면 구조만 읽고 넘어가도 됩니다.


In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    model_name = os.environ.get("LANGCHAIN_MODEL", "openai:gpt-5.4")
    agent = create_agent(
        model=model_name,
        tools=[multiply],
        system_prompt="너는 필요한 경우 tool을 사용해 정확하게 계산하는 assistant야.",
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "7 곱하기 6은 얼마야?"}]}
    )
    print(result["messages"][-1].content)
else:
    print("OPENAI_API_KEY가 없어서 agent 실행은 건너뜁니다. 위의 tool 셀만으로도 tool 객체의 기본은 확인했습니다.")


## 19. Config와 Debugging 기초

LangChain Runnable은 실행할 때 config를 붙일 수 있습니다. tags와 metadata는 LangSmith 같은 tracing 도구에서 실행을 구분할 때 유용합니다.

처음에는 tracing을 켜지 않아도 되지만, chain이 길어질수록 어느 단계에서 문제가 생겼는지 추적하는 습관이 중요합니다.


In [ ]:
debug_model = FakeListChatModel(responses=["config는 실행에 태그와 메타데이터를 붙일 때 유용합니다."])
debug_chain = explain_prompt | debug_model | StrOutputParser()

print(
    debug_chain.with_config(
        {
            "tags": ["week02", "langchain-basics"],
            "metadata": {"lesson": "config"},
        }
    ).invoke({"concept": "Runnable config"})
)


## 20. 처음 배울 때 자주 헷갈리는 점

### Chain과 Agent는 다릅니다

Chain은 정해진 순서로 실행되는 파이프라인입니다. Agent는 모델이 tool 사용 여부와 다음 행동을 선택합니다. 단순한 RAG라면 chain으로 충분한 경우가 많고, 여러 tool을 상황에 따라 써야 하면 agent를 고려합니다.

### Prompt만 잘 써도 부족할 수 있습니다

RAG에서는 prompt보다 검색 품질이 병목인 경우가 많습니다. 엉뚱한 문서를 가져오면 모델이 아무리 좋아도 답이 흔들립니다.

### Embedding과 Vector Store는 같은 것이 아닙니다

Embedding model은 텍스트를 벡터로 바꾸는 모델입니다. Vector store는 그 벡터를 저장하고 검색하는 시스템입니다.

### Retriever는 꼭 vector search만 뜻하지 않습니다

Retriever는 질문을 받아 문서를 반환하는 인터페이스입니다. Vector store 기반일 수도 있고, BM25, 웹 검색, 데이터베이스 검색일 수도 있습니다.

### LangChain import 경로는 버전에 민감합니다

1.x에서는 `langchain_core`, `langchain_community`, provider별 패키지로 역할이 많이 나뉘어 있습니다. 오래된 블로그의 `LLMChain` 예제는 현재 권장 흐름과 다를 수 있습니다.


## 21. 직접 해볼 연습문제

아래 문제를 직접 바꿔 가며 실행해 보세요.

1. `explain_prompt`의 system message를 더 엄격하게 바꿔 보세요.
2. `JsonOutputParser`의 `StudyPlan`에 `estimated_minutes` 필드를 추가해 보세요.
3. `rag_docs`에 문서를 2개 더 추가하고 검색 결과가 어떻게 바뀌는지 보세요.
4. `vocabulary`에 한국어 단어를 추가하고 한국어 query로 검색해 보세요.
5. `chunk_size`와 `chunk_overlap`을 바꿔 chunk 결과를 비교해 보세요.
6. `multiply` 말고 `calculate_tax(price, rate)` tool을 만들어 보세요.
7. API 키가 있다면 fake model을 실제 model로 바꾸고 같은 chain을 실행해 보세요.


## 22. 학습 체크리스트

아래 질문에 답할 수 있으면 LangChain 기초 흐름은 잡힌 것입니다.

- `PromptTemplate`과 `ChatPromptTemplate`의 차이를 설명할 수 있는가?
- `prompt | model | parser`에서 각 단계의 입력과 출력이 무엇인지 말할 수 있는가?
- `invoke`, `batch`, `stream`의 차이를 설명할 수 있는가?
- `Document.page_content`와 `Document.metadata`의 역할을 구분할 수 있는가?
- splitter가 필요한 이유를 말할 수 있는가?
- embedding model과 vector store의 차이를 설명할 수 있는가?
- retriever가 RAG에서 어디에 들어가는지 말할 수 있는가?
- chain과 agent를 언제 다르게 쓰는지 설명할 수 있는가?

## 참고한 공식 문서

- LangChain Quickstart: https://docs.langchain.com/oss/python/langchain/quickstart
- Models: https://docs.langchain.com/oss/python/langchain/models
- Chat model integrations: https://docs.langchain.com/oss/python/integrations/chat/
- Text splitters: https://docs.langchain.com/oss/python/integrations/splitters/index
- Vector stores: https://docs.langchain.com/oss/python/integrations/vectorstores/
- Retrievers: https://docs.langchain.com/oss/python/integrations/retrievers/index
- Runnable / LCEL API: https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.base.Runnable.html
